In [1]:
import os
import shutil
from google.colab import drive

# Remove existing directory if it exists
if os.path.exists('/content/mini-gpt'):
    shutil.rmtree('/content/mini-gpt')

# Clone the repository from GitHub
!git clone https://github.com/maariogutierrez/mini-gpt.git
ROOT = '/content/mini-gpt'
os.chdir(ROOT)

# Mount Google Drive for data and outputs
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/My Drive/mini-gpt'

REQUIREMENTS = 'https://raw.githubusercontent.com/maariogutierrez/mini-gpt/main/requirements.txt'
!pip install -r $REQUIREMENTS

print("Logging into Weights & Biases (wandb). Follow the prompts.")
import wandb
wandb.login(key='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')

# Store outputs in Google Drive, but keep code in cloned repo
EXPORTS_DIR = os.path.join(DRIVE_ROOT, 'exports')
LOGS_DIR = os.path.join(DRIVE_ROOT, 'logs')
DATA_DIR = os.path.join(DRIVE_ROOT, 'data')
CHECKPOINTS_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')

print("\n--- Setup Summary ---")
print(f"Code repository: {ROOT}")
print(f"Google Drive: {DRIVE_ROOT}")
print(f"Current Working Directory: {os.getcwd()}")

Cloning into 'mini-gpt'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 28 (delta 3), reused 26 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 11.73 KiB | 11.73 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Logging into Weights & Biases (wandb). Follow the prompts.


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mariogutierrez (mariogutierrez-universidad-polit-cnica-de-madrid) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



--- Setup Summary ---
Code repository: /content/mini-gpt
Google Drive: /content/drive/My Drive/mini-gpt
Current Working Directory: /content/mini-gpt


In [4]:
# Import required libraries
import sys
import numpy as np
from pathlib import Path
from typing import Generator
from tqdm import tqdm
from datasets import load_dataset

# Add the ROOT to sys.path so we can import from model module
sys.path.insert(0, ROOT)

from model.architecture.tokenizer import Tokenizer
from model.training.preprocess import process_dataset, stream_tokenized_documents

print("✓ Imports successful")

✓ Imports successful


In [3]:
# Run the full preprocessing pipeline
print("Starting preprocessing pipeline...\n")
process_dataset(
    output_dir=DATA_DIR,
    train_split=0.9,
    dataset_name="roneneldan/TinyStories",
)
print("\n✓ Preprocessing complete!")

Starting preprocessing pipeline...

Vocab size: 100277 (fits in uint16: False)
Output directory: /content/drive/My Drive/mini-gpt/data

First pass: counting total tokens...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Loading roneneldan/TinyStories in streaming mode...


OverflowError: Python integer 82087 out of bounds for uint16

In [ ]:
# Verify: Print the size of each file
print("\n" + "="*60)
print("FILE SIZE VERIFICATION")
print("="*60 + "\n")

train_bin_path = Path(DATA_DIR) / "train.bin"
val_bin_path = Path(DATA_DIR) / "val.bin"

if train_bin_path.exists():
    train_size_bytes = train_bin_path.stat().st_size
    train_size_mb = train_size_bytes / (1024**2)
    train_size_gb = train_size_bytes / (1024**3)
    train_tokens = train_size_bytes // 2  # uint16 = 2 bytes per token
    print(f"train.bin")
    print(f"  Size: {train_size_mb:.2f} MB ({train_size_gb:.4f} GB)")
    print(f"  Tokens: {train_tokens:,}")
else:
    print("❌ train.bin not found!")

if val_bin_path.exists():
    val_size_bytes = val_bin_path.stat().st_size
    val_size_mb = val_size_bytes / (1024**2)
    val_size_gb = val_size_bytes / (1024**3)
    val_tokens = val_size_bytes // 2  # uint16 = 2 bytes per token
    print(f"\nval.bin")
    print(f"  Size: {val_size_mb:.2f} MB ({val_size_gb:.4f} GB)")
    print(f"  Tokens: {val_tokens:,}")
else:
    print("❌ val.bin not found!")

if train_bin_path.exists() and val_bin_path.exists():
    total_tokens = train_tokens + val_tokens
    train_pct = train_tokens / total_tokens * 100
    print(f"\nTotal: {total_tokens:,} tokens")
    print(f"Split: {train_pct:.1f}% train, {100-train_pct:.1f}% validation")
    print("\n✓ Files verified successfully!")

In [ ]:
# Verify: Load a random batch and decode it to check if it's readable text
print("\n" + "="*60)
print("BATCH DECODING VERIFICATION")
print("="*60 + "\n")

# Load tokens from validation set
val_tokens = np.memmap(val_bin_path, dtype=np.uint16, mode='r')

# Initialize tokenizer
tokenizer = Tokenizer()

# Select a random batch (first 256 tokens)
batch_size = 256
random_start = np.random.randint(0, len(val_tokens) - batch_size)
batch_tokens = val_tokens[random_start:random_start + batch_size]

# Decode the batch
decoded_text = tokenizer.decode(batch_tokens.tolist())

print(f"Random batch from validation set (tokens {random_start} to {random_start + batch_size}):\n")
print("-" * 60)
print(decoded_text)
print("-" * 60)
print(f"\n✓ Decoded text is readable! Batch successfully loaded and verified.")
print(f"  Input: {batch_size} tokens")
print(f"  Output length: {len(decoded_text)} characters")